# Estimating Power Plant Capacity Factors from Plant Characteristics
**A CRISP-DM analysis of the WRI Global Power Plant Database (v1.3.0)**

Author: Philip Dawson-Gariepy · Module: Data Analytics (CA2) · Munster Technoligical University

## 0. Setup
This section imports the libraries used throughout the notebook, records software versions for repoducibility, and fixes a random seed so results can be repeated.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

RANDOM_STATE = 42 # fixed seed used for every split and model
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 50)

print("Python       ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)
print("seaborn     ", sns.__version__)
print("scikit-learn", sklearn.__version__)

Python        3.14.7
pandas       3.0.6
numpy        2.5.4
matplotlib   3.11.2
seaborn      0.13.2
scikit-learn 1.9.1


### Load the raw data
This raw CSV is not stored in the repository. See `data_instructions/README.md` for how to download it and where to place it. The path below is relative to the `notebooks/` folder. The shape check confirms the expected v1.3.0 file was loaded.

In [2]:
DATA_PATH = "../data/global_power_plant_database.csv"

raw = pd.read_csv(DATA_PATH, low_memory=False)
assert raw.shape == (34936, 36), f"Unexpected shape: {raw.shape} - check data_instrcutions/README.md"

df = raw.copy() # work on a copy; `raw` is never modified
df.head()

,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,other_fuel2,other_fuel3,commissioning_year,owner,source,url,geolocation_source,wepp_id,year_of_capacity_data,generation_gwh_2013,generation_gwh_2014,generation_gwh_2015,generation_gwh_2016,generation_gwh_2017,generation_gwh_2018,generation_gwh_2019,generation_data_source,estimated_generation_gwh_2013,estimated_generation_gwh_2014,estimated_generation_gwh_2015,estimated_generation_gwh_2016,estimated_generation_gwh_2017,estimated_generation_note_2013,estimated_generation_note_2014,estimated_generation_note_2015,estimated_generation_note_2016,estimated_generation_note_2017
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,GEODB0040538,33.0,32.322,65.1190,Hydro,NaN,NaN,NaN,NaN,NaN,GEODB,http://globalenergyobservatory.org,GEODB,1009793,2017.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,123.77,162.90,97.39,137.76,119.50,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
1,AFG,Afghanistan,Kandahar DOG,WKS0070144,10.0,31.670,65.7950,Solar,NaN,NaN,NaN,NaN,NaN,Wiki-Solar,https://www.wiki-solar.org,Wiki-Solar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.43,17.48,18.25,17.70,18.29,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
2,AFG,Afghanistan,Kandahar JOL,WKS0071196,10.0,31.623,65.7920,Solar,NaN,NaN,NaN,NaN,NaN,Wiki-Solar,https://www.wiki-solar.org,Wiki-Solar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.64,17.58,19.10,17.62,18.72,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,GEODB0040541,66.0,34.556,69.4787,Hydro,NaN,NaN,NaN,NaN,NaN,GEODB,http://globalenergyobservatory.org,GEODB,1009795,2017.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,225.06,203.55,146.90,230.18,174.91,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,GEODB0040534,100.0,34.641,69.7170,Hydro,NaN,NaN,NaN,NaN,NaN,GEODB,http://globalenergyobservatory.org,GEODB,1009797,2017.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,406.16,357.22,270.99,395.38,350.80,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1


---
# Phase 1: Business Understanding

## 1.0 Background: how power plants work (for non-specialists)
This project uses some electrical engineering ideas. This section explains them in everyday terms, so the rest of the notebook can be followed without an engineering background.

### Power versus energy
A kettle might be rated at **2 kilowatts (kW)**. That is its *power*: how fast it can use electricity. If it runs for half an hour, it uses **1 kilowatt-hour (kWh)** of *energy*. Power plants work the same way, on a much larger scale:

| Term | Meaning | Kettle example | Power plant example |
| --- | --- | --- | --- |
| **Capacity** (MW, megawatts) | The most power the plant can produce at any moment: its "rating" | 2 kW | A 100 MW wind farm |
| **Generation** (GWh, gigawatt-hours) | The energy the plant actually produced over a period | 1 kWh in half an hour | 219 GWh in a year |

1 MW = 1,000 kW, and 1 GWh = 1,000,000 kWh. A 100 MW plant running flat out for one hour produces 100 MWh (0.1 GWh).

### Capacity factor: the key idea
No plant runs at full power all year. The **capacity factor** measures how hard a plant actually worked, compared with the most it could have produced:

```
capacity factor = energy actually produced ÷ energy it could have produced at full power all year
```

A year has 8,760 hours, so a 100 MW plant could produce at most 100 × 8,760 = 876,000 MWh, or **876 GWh**. If it actually produced 219 GWh, its capacity factor is 219 ÷ 876 = **0.25**: on average it ran at a quarter of its maximum. A capacity factor always lies between 0 (never ran) and 1 (ran at full power every hour of the year). In practice even the most heavily used plants stay below about 0.9, because they must stop for maintenance [1].

### Why different plants run so differently
How hard a plant runs depends mostly on what kind of plant it is. The US Energy Information Administration describes the same broad patterns [2]:

| Plant type | How it behaves | What to expect |
| --- | --- | --- |
| **Nuclear** | Runs almost non-stop. Fuel is cheap but stopping and restarting is slow and costly, so it stays on except for refuelling and maintenance. | Very high |
| **Coal** | Built to run for long periods, but used less when cheaper electricity is available. | Medium to high |
| **Gas** | Two very different kinds share this label. *Combined-cycle* plants are efficient and run a lot; *gas turbines* ("peakers") start quickly and run only when demand is highest. The dataset does not say which kind a plant is. | Anywhere from very low to high, so hard to predict |
| **Oil** | Mostly backup or emergency plants, used rarely. | Very low |
| **Hydro** | Depends on rain, river flow and snowmelt, which vary by region and year. | Medium |
| **Wind** | Only produces when the wind blows; cannot be switched on at will. | Low to medium |
| **Solar** | Only produces in daylight; output depends on how far from the equator the plant is and how cloudy it is. | Low |
| **Geothermal, biomass, waste** | Fuel or heat is available continuously, so these run steadily. | Medium to high |
| **Storage** (batteries, pumped hydro) | Stores electricity and releases it later, losing some along the way, so over a year it uses more than it produces. | Not meaningful, so excluded |

### The grid decides who runs
Demand for electricity rises and falls through the day. Grid operators generally call on the cheapest available plants first and bring in more expensive ones only when demand is high (often called the *merit order*). This means **how hard a plant runs depends partly on the system around it**: fuel prices, other plants on the same grid, and national policy. None of that information is in the dataset, which limits how accurate any estimate can be.

### Why some values are negative
Plants use some of their own electricity to run pumps, fans, lighting and controls. Published figures are usually *net* generation: output minus the plant's own use. A backup plant that barely runs during a year can therefore report **negative** net generation. This explains some of the negative values found in the data.

### What this means for the model
The engineering background suggests which plant characteristics should carry useful information:
- **Fuel type**: the strongest expected signal, for the reasons in the table above.
- **Latitude**: plants closer to the equator receive more sunlight, which matters most for solar.
- **Commissioning year (age)**: older plants are often less efficient and may be called on less; newer wind and solar technology tends to perform better.
- **Capacity**: plant size can hint at its role in the grid, for example small backup units versus large baseload stations.
- **Country**: a rough stand-in for climate, the national grid mix and energy policy.

## 1.1 Context and problem
To understand a power plant's impact on the environment, we need to know how much electricity it actually produced, not just how big it is. Two plants of the same size can have very different emissions if one runs all year and the other only on the coldest winter evenings.

This information is surprisingly hard to find. Most countries do not publish how much electricity each individual plant produces, including many developing countries where electricity production is expected to grow fastest [1]. The dataset used here, the Global Power Plant Database [3], shows the gap clearly:

- It lists **34,936 power plants** in 167 countries.
- Only **9,500** of them (27%) report their 2017 generation, and **8,370** of those are in the United States.
- The plants that do **not** report make up about **two-thirds of the world's generating capacity** in the database.

**The problem:** for most of the world's power plants, we know their size, fuel and location, but not how much electricity they produced. Anyone estimating national electricity output or emissions has to fill that gap with estimates, and it is unclear how trustworthy those estimates are.

## 1.2 Stakeholders and users
| Role | Who | What they care about |
| --- | --- | --- |
| Data suppliers | The World Resources Institute (WRI) and partners, who gathered the data from national sources such as the US Energy Information Administration, India's Central Electricity Authority and Australia's Clean Energy Regulator | That the data is accurate and openly reused |
| **Primary user** | **Analysts who estimate how much electricity, and therefore how much pollution, power plants produce in countries that do not publish this** (for example climate research groups and organisations tracking emissions from the energy sector) | Estimates they can rely on, and a clear warning where they cannot |
| Secondary users | Energy planners and researchers comparing countries' power systems | Figures that are comparable between countries |
| Affected parties | Countries and plant owners whose output or emissions may be over- or under-estimated | Being represented fairly; wrong estimates could distort policy debates or damage reputations |

Throughout the project, the focus is on the primary user: someone with a list of plants whose size, fuel and location are known but whose actual output is not, who needs a sensible estimate for each one.

## 1.3 Practical objective
The aim is to help analysts **fill in missing power plant output figures with estimates whose accuracy is known**. That means the analyst can produce output and emissions figures for plants that do not report them, and can tell which estimates are reliable enough to use and which should be treated with caution.

The result supports human analysis and reporting. It is not meant to make automatic decisions.

## 1.4 Analytics objective
Build and evaluate a **regression model** that estimates a plant's **2017 annual capacity factor** from characteristics available for most plants (primary fuel, capacity, commissioning year, location), and measure its error on plants it has not seen.

- **Unit of analysis:** one power plant.
- **Target:** capacity factor, constructed as
  `capacity_factor = generation_gwh_2017 / (capacity_mw × 8.76)`
  where 8.76 converts MW to the GWh a plant would produce running at full output for a year (8,760 hours ÷ 1,000). A valid value lies between 0 and 1.
- **Why capacity factor rather than raw generation:** generation is dominated by plant size, so a model predicting GWh would mostly learn that big plants generate more, and its errors would be driven by the few largest plants. Capacity factor measures how intensively a plant runs independent of its size, so a 2 MW and a 2,000 MW plant can be compared directly. Yin et al. (2020) make the same choice. Predicted capacity factor converts back to generation by multiplying by capacity × 8.76.
- **Year:** 2017 is used because it is the most recent year with both reported generation and WRI's own published estimates, which serve as a benchmark.

## 1.5 Success criteria
**Technical criteria** (measured on a held-out test set):

| ID | Criterion | Measure |
| --- | --- | --- |
| T1 | The model beats a simple baseline: the median capacity factor of the plant's fuel type, learned from training data | Lower mean absolute error (MAE) than the baseline |
| T2 | The model is compared fairly with WRI's published 2017 estimates on the same test plants | MAE per fuel type, model vs. WRI |
| T3 | Error is reported in practical units | MAE in capacity-factor points and in GWh |
| T4 | Performance is checked on US and non-US plants separately | MAE for each group |

**Practical criteria:**

| ID | Criterion |
| --- | --- |
| P1 | Overall MAE of **0.10 capacity-factor points or less**. For a median-sized plant (about 18 MW) this is roughly ±15 GWh a year; for a 100 MW plant, about ±88 GWh. |
| P2 | The main drivers of the estimate can be explained in plain language to a non-technical user. |
| P3 | The fuel types and regions where estimates are unreliable are identified, so a user knows where not to rely on them. |

**Preliminary feasibility check.** On the 9,275 plants with a valid 2017 capacity factor (storage excluded, values outside 0–1 removed), the fuel-median baseline has an MAE of about 0.14, while WRI's published estimates have an MAE of about 0.17. WRI is more accurate for solar, wind, hydro and nuclear, but much less accurate for gas and waste. This baseline figure is in-sample and therefore optimistic; it will be recalculated on a proper train/test split in Phase 4. It shows that P1 requires a meaningful improvement over the baseline (around 30%) and that some fuels, notably gas, are likely to remain hard to estimate. P1 may be revised after Data Understanding, and any revision will be recorded.

The cell below reproduces the preliminary feasibility figures quoted above.

In [5]:
# Preliminary feasibility check (in-sample; recalculated properly in Phase 4)
chk = df[df["generation_gwh_2017"].notna() & (df["primary_fuel"] != "Storage")].copy()
chk["cf"] = chk["generation_gwh_2017"] / (chk["capacity_mw"] * 8.76)
chk = chk[chk["cf"].between(0, 1)]
chk["wri_cf"] = chk["estimated_generation_gwh_2017"] / (chk["capacity_mw"] * 8.76)

baseline = chk.groupby("primary_fuel")["cf"].transform("median")
has_wri = chk["wri_cf"].notna()

print(f"Plants with valid 2017 capacity factor: {len(chk):,}")
print(f"Fuel-median baseline MAE: {(chk['cf'] - baseline).abs().mean():.3f}")
print(f"WRI estimate MAE:         {(chk.loc[has_wri, 'cf'] - chk.loc[has_wri, 'wri_cf']).abs().mean():.3f}")

Plants with valid 2017 capacity factor: 9,275
Fuel-median baseline MAE: 0.141
WRI estimate MAE:         0.171


## 1.6 Constraints and assumptions
**Constraints**
- **Data currency:** the database's final release is v1.3.0 (June 2021) and is no longer maintained, so findings describe the power system around 2017.
- **Representation:** about 88% of plants with reported generation are in the US, so the model will largely learn US plant behaviour, while estimates are most needed elsewhere.
- **Missing predictors:** factors that strongly influence how often a plant runs, such as fuel prices, grid dispatch decisions, plant efficiency and technology, are not in the data (Yin et al., 2020, Section 8).
- **Time and resources:** an eight-week project using standard Python tools, which favours a simple, well-evaluated model over a complex one.
- **Privacy:** the data contains no personal information. The `owner` field names companies and is not used for modelling.

**Assumptions**
- Plant characteristics carry enough signal to estimate capacity factor better than fuel type alone.
- Reported generation is broadly accurate after removing physically impossible values; the data source's own reliability cannot be independently verified.
- Storage plants are out of scope, because they consume electricity by design and capacity factor does not mean the same thing for them.

## 1.7 Initial project plan
| Week | CRISP-DM phase | Main activities |
| --- | --- | --- |
| 6 | Figure Dataset | Decide problem, data, unit of analysis, method and risks |
| 7 | Business Understanding | Objectives, success criteria, plan; repository and README |
| 8 | Data Understanding | Collection, description, exploration and quality reports; data dictionary |
| 9 | Data Preparation | Selection, cleaning, target construction, train/test split, preparation log |
| 10 | Modelling | Fuel-median baseline, linear regression, decision tree regressor |
| 11 | Evaluation | Test-set metrics, comparison with WRI, US vs. non-US analysis; paper draft |
| 12 | Deployment and submission | Recommendations, reflection, AI statement, final paper |

**Proposed technique:** regression, starting from the fuel-median baseline, then multiple linear regression (interpretable), then a decision tree regressor (captures non-linear effects and fuel-specific behaviour). The choice of final model depends on test-set performance against the criteria above.

---
# Phase 2: Data Understanding

## 2.1 Initial Data Collection Report

## 2.2 Data Description Report

## 2.3 Data Exploration Report

## 2.4 Data Quality Report

---
# Phase 3: Data Preparation


## 3.1 Preparation log
| Issue | Action | Rationale / evidence | Assumption | Number affected | Risk / verification |
| --- | --- | --- | --- | --- | --- |
| | | | | | |

## 3.2 Selection

## 3.3 Cleaning

## 3.4 Type and format correction

## 3.5 Feature construction

## 3.6 Train/test split, encoding and scaling

## 3.7 Validation

---
# Phase 4: Modelling

## 4.1 Task, inputs and target

## 4.2 Baseline

## 4.3 Primary model

## 4.4 Alternative model or setting

## 4.5 Interpretability and limitations

---
# Phase 5: Evaluation

## 5.1 Results against technical success criteria

## 5.2 Practical usefulness

## 5.3 Generalisability, error analysis and failure cases

## 5.4 Fairness, transparency and misuse

## 5.5 Decision

---
# Phase 6: Deployment and Recommendations

---
# Reflection
*What changed from the original plan? Which decision needed the most judgement? What failed and what was learned? What would you do with more time or better data? Which parts can you explain confidently without assistance?*

# Responsible AI Collaboration Statement
*300–500 words. Must address all of the following:*
1. *Tools and versions used.*
2. *Tasks for which AI assistance was used.*
3. *One or more representative prompts, or a link to a prompt log.*
4. *How code, factual claims, calculations and references were verified.*
5. *Examples of outputs that were rejected, corrected or substantially modified.*
6. *What remained your own judgement and decision.*
7. *What you learned about the strengths and limitations of AI collaboration.*

# References
[1] L. Yin, L. Byers, L. Malaguzzi Valeri, and J. Friedrich, "Estimating power plant generation in the Global Power Plant Database," World Resources Institute, Washington, DC, USA, Tech. Note, Mar. 2020. [Online]. Available: https://www.wri.org/research/estimating-power-plant-generation-global-power-plant-database (accessed Nov. 5, 2026).

[2] U.S. Energy Information Administration, "Monthly generator capacity factor data now available by fuel and technology," *Today in Energy*, Jan. 15, 2014. [Online]. Available: https://www.eia.gov/todayinenergy/detail.php?id=14611 (accessed Nov. 5, 2026).

[3] Global Energy Observatory, Google, KTH Royal Institute of Technology in Stockholm, Enipedia, and World Resources Institute, *Global Power Plant Database*, version 1.3.0, World Resources Institute, Jun. 2021. [Dataset]. Available: https://datasets.wri.org/dataset/globalpowerplantdatabase (accessed Oct. 9, 2026).